# Dataset-MoE NIDS — complete Colab training and evaluation

Run this notebook from top to bottom. It securely clones the private GitHub repository using a token held in **Colab Secrets**, mounts the NIDS datasets from Google Drive, validates the environment, runs Stage A → B → C, evaluates the test split, writes resumable checkpoints/results to Drive, and displays the resulting metrics.

Before starting:

1. In Colab choose **Runtime → Change runtime type → GPU**.
2. Open the key icon (**Secrets**) and add `GITHUB_TOKEN`. Grant this notebook access. Use a fine-grained token with **Contents: Read-only** access to this private repository. Never paste a token into a notebook cell.
3. Put the datasets under `MyDrive/NIDS_datasets/` using the directory names expected by `data/registry.py`.
4. Edit only the configuration cell below, then use **Runtime → Run all**.


In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"

DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"
RUN_NAME = "colab_dataset_moe_nids"

# "smoke" checks the complete pipeline on at most 5,000 rows/dataset
# for one epoch per stage. Change to "full" for the real experiment.
RUN_MODE = "smoke"  # "smoke" | "full"

# The default performs the primary experiment. Add variants only when
# intentionally running a comparison; each receives separate checkpoints.
ARCHITECTURES = ["moe_dataset_soft"]
# Valid: moe_dataset_soft, moe_dataset_hard_gate, moe_dataset_adapters,
#        plain_pooled, no_fusion, hard_two_stage

# None uses every dataset in config/default.yaml. A short list is useful
# for development, e.g. ["NF-UNSW-NB15-v3", "NF-BoT-IoT-v3"].
ACTIVE_DATASETS = None

FORCE_RESTART = False   # False resumes/skips completed MoE stages
RUN_TESTS = True
# ==================================================================


## 1. Secure repository checkout and environment setup

The token is read at runtime and supplied to Git through a temporary askpass helper. It is not printed, placed in the clone URL, saved in notebook output, or written into Git configuration. If the secret is unavailable, Colab prompts for it without echoing the value.


In [ ]:
import getpass
import os
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended to run in Google Colab.") from exc

drive.mount("/content/drive")

try:
    github_token = userdata.get(GITHUB_SECRET_NAME)
except Exception:
    github_token = getpass.getpass(f"Enter {GITHUB_SECRET_NAME} (input is hidden): ")
if not github_token:
    raise RuntimeError(f"Missing GitHub token. Add {GITHUB_SECRET_NAME} in Colab Secrets.")

repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
askpass = Path("/tmp/dataset_moe_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\nesac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
    "GITHUB_TOKEN": github_token,
})

def git(*args):
    subprocess.run(["git", *args], check=True, env=git_env)

try:
    if (repo_dir / ".git").is_dir():
        git("-C", str(repo_dir), "remote", "set-url", "origin", repo_url)
        git("-C", str(repo_dir), "fetch", "--prune", "origin", GITHUB_BRANCH)
        git("-C", str(repo_dir), "checkout", GITHUB_BRANCH)
        git("-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH)
    else:
        if repo_dir.exists() and any(repo_dir.iterdir()):
            raise RuntimeError(f"{repo_dir} exists and is not an empty Git repository.")
        git("clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir))
finally:
    github_token = None
    git_env.pop("GITHUB_TOKEN", None)
    askpass.unlink(missing_ok=True)

os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Preflight checks

This fails early if the GPU is unavailable, an architecture name is invalid, or an active dataset path is missing. The smoke mode is still a real end-to-end run and is the recommended first execution.


In [ ]:
import torch
from data.registry import get_spec
from training.config import load_config
from training.run import ALL_ARCHITECTURES

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
unknown = sorted(set(ARCHITECTURES) - set(ALL_ARCHITECTURES))
if unknown:
    raise ValueError(f"Unknown architectures: {unknown}; valid values: {ALL_ARCHITECTURES}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU, reconnect, and Run all.")

preview_overrides = []
if ACTIVE_DATASETS:
    preview_overrides.append("data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]")
preview = load_config("config/default.yaml", preview_overrides)
missing = {}
for dataset_name in preview["data"]["active_datasets"]:
    paths = get_spec(dataset_name).paths
    if not any(Path(path).exists() for path in paths):
        missing[dataset_name] = paths
if missing:
    details = "\n".join(f"  {name}: {paths}" for name, paths in missing.items())
    raise FileNotFoundError(f"Missing dataset files/directories under {DRIVE_DATA_DIR}:\n{details}")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Datasets:", preview["data"]["active_datasets"])
print("Architectures:", ARCHITECTURES, "| mode:", RUN_MODE)
print("Drive outputs:", DRIVE_OUTPUT_DIR)


In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Tests skipped by configuration.")


## 3. Train and evaluate

Each MoE run executes Stage A, Stage B, and Stage C through the repository's canonical CLI, then performs held-out test evaluation and bootstrap confidence intervals. Progress checkpoints are copied to Drive after every completed stage, so rerunning after a disconnected Colab session resumes safely when `FORCE_RESTART` is `False`. Baseline architectures use their corresponding end-to-end training path.


In [ ]:
from datetime import datetime

completed_runs = []
for architecture in ARCHITECTURES:
    variant_run_name = RUN_NAME if len(ARCHITECTURES) == 1 else f"{RUN_NAME}_{architecture}"
    cmd = [
        sys.executable, "-m", "training.run",
        "--config", "config/default.yaml",
        "--mode", RUN_MODE,
        "--set", f"run_name={variant_run_name}",
        "--set", f"architecture={architecture}",
        "--set", "training.device=cuda",
        "--set", f"training.force_restart={str(FORCE_RESTART).lower()}",
    ]
    if ACTIVE_DATASETS:
        cmd += ["--set", "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]"]

    print(f"\n[{datetime.now().isoformat(timespec='seconds')}] Starting {architecture} ({RUN_MODE})")
    subprocess.run(cmd, check=True, env=os.environ.copy())
    completed_runs.append({
        "architecture": architecture,
        "run_name": variant_run_name,
        "result_dir": str(Path(DRIVE_OUTPUT_DIR) / "results" / variant_run_name),
        "checkpoint_dir": str(Path(DRIVE_OUTPUT_DIR) / "checkpoints" / variant_run_name),
    })

print("\nAll requested training and evaluation runs completed.")


## 4. Review persisted results

The tables below are read back from Google Drive, confirming that the durable artifacts—not just transient notebook variables—were written successfully. `Overall_Metrics.csv`, `Per_Class_Metrics.csv`, and `Per_Dataset_Metrics.csv` are keyed by `Trial_ID`.


In [ ]:
import pandas as pd
from IPython.display import display

for run in completed_runs:
    result_dir = Path(run["result_dir"])
    print(f"\n=== {run['architecture']} ===")
    for filename in ["Overall_Metrics.csv", "Per_Dataset_Metrics.csv", "Per_Class_Metrics.csv"]:
        path = result_dir / filename
        if not path.is_file():
            raise FileNotFoundError(f"Expected result was not persisted: {path}")
        frame = pd.read_csv(path)
        print(filename, "—", path)
        display(frame.tail(30))
    print("Checkpoints:", run["checkpoint_dir"])


## Operational notes

- Start with `RUN_MODE = "smoke"`. After it completes, change to `"full"` and use a new `RUN_NAME`.
- Keep `FORCE_RESTART = False` to resume after a runtime disconnect. Set it to `True` only when deliberately discarding progress for the same run name.
- Training reads datasets from Drive but uses `/content` as scratch space for frequent checkpoint operations. Completed stage artifacts are synchronized back to Drive.
- Do not commit dataset files, tokens, generated checkpoints, or result directories to Git. The repository `.gitignore` excludes these paths.
